# EKATENA – Registro, censura y extracción de datos

El proceso tiene **3 secciones**, que se ejecutan en orden:

| Sección | Qué hace | Estado del documento |
|---|---|---|
| **1. Registro** | Lee los PDF de la carpeta `entrada/`, les asigna un ID (`DOC-000001`, ...) y los guarda en la base de datos `ekatena.db`. | `registrado` |
| **2. Censura** | Tapa en negro los textos de `TEXTOS_A_CENSURAR` y guarda la copia en `censurados/DOC-000001.pdf`. | `censurado` (o `revisar`) |
| **3. Extracción** | Envía el PDF **ya censurado** a Claude, que extrae los campos de `CAMPOS` y los guarda en la base de datos. | `extraido` |

Cada sección solo procesa los documentos que están en el estado anterior, así que puedes ejecutarlas varias veces sin repetir trabajo.

**Antes de empezar (solo la primera vez)**
1. Instala las librerías: ejecuta en una celda `%pip install pypdf pdfplumber pytesseract pypdfium2 pillow anthropic` y pulsa **Restart**.
2. Instala **Tesseract OCR** (para PDF escaneados) con el idioma Spanish.
3. Para la sección 3 necesitas una **API key de Anthropic** guardada en la variable de entorno `ANTHROPIC_API_KEY` (ver la sección 3). Nunca la escribas dentro del notebook.

**Privacidad:** el original nunca sale de tu equipo. A Claude solo se envía el PDF censurado, y el archivo censurado se nombra con el ID (no con el nombre original, que puede incluir el nombre de la empresa).

In [ ]:
# === Configuración ===
import os
from ekatena import base_datos as bd
from ekatena.pdf_ocr import contar_paginas
from ekatena.censura import censurar_pdf, verificar_censura

CARPETA_ENTRADA = "entrada"        # aquí pones los PDF escaneados
CARPETA_CENSURADOS = "censurados"  # aquí quedan las copias censuradas
BASE_DATOS = "ekatena.db"

# Ruta de tesseract.exe solo si no se encuentra solo. Si no, déjalo vacío: r""
TESSERACT_CMD = r""

# Textos a tapar en negro. Añade variantes si hace falta, por ejemplo "DKDA" solo.
TEXTOS_A_CENSURAR = [
    "CORPORATIVO AMBIENTAL DKDA",
]
MARGEN = 2              # borde extra alrededor de cada texto censurado (en puntos)
DPI_SALIDA = 200        # calidad del PDF censurado
VERIFICAR_CENSURA = True  # vuelve a leer el PDF censurado para comprobar que no quedó nada

# Campos que Claude debe extraer: "nombre_del_campo": "descripción para Claude"
CAMPOS = {
    "numero_reporte": "Número o folio del reporte, p. ej. EKT-00674515B8",
    "fecha_reporte": "Fecha del reporte que aparece arriba a la derecha",
    "fecha_creacion": "Fecha y hora de 'Creación'",
    "fecha_ultima_actualizacion": "Fecha y hora de 'Última Actualización'",
    "avance": "Porcentaje de 'Avance'",
    "calificacion": "Calificación numérica principal (el número grande junto a la barra de colores)",
    "escala": "Escala de la calificación, p. ej. 0-895",
}

os.makedirs(CARPETA_ENTRADA, exist_ok=True)
os.makedirs(CARPETA_CENSURADOS, exist_ok=True)
con = bd.conectar(BASE_DATOS)


def mostrar_documentos():
    filas = bd.tabla_documentos(con)
    if not filas:
        print("(la base de datos está vacía)")
        return
    print(f"{'ID':<11} {'ESTADO':<11} {'PÁG':>4} {'ZONAS':>5}  ARCHIVO")
    for f in filas:
        zonas = "" if f["zonas_censuradas"] is None else f["zonas_censuradas"]
        print(f"{f['id']:<11} {f['estado']:<11} {f['paginas'] or '':>4} {zonas:>5}  {f['nombre_archivo']}")
        if f["notas"]:
            print(f"{'':<11} -> {f['notas']}")


print(f"Base de datos: {os.path.abspath(BASE_DATOS)}")
mostrar_documentos()

## Sección 1 – Registrar los escaneados en la base de datos

Copia tus PDF a la carpeta `entrada/` y ejecuta la celda. Cada PDF nuevo recibe un ID. Si el mismo archivo ya estaba registrado (aunque le cambies el nombre), no se duplica.

In [ ]:
# === Sección 1: registro ===
pdfs = sorted(f for f in os.listdir(CARPETA_ENTRADA) if f.lower().endswith(".pdf"))
if not pdfs:
    print(f"No hay PDF en la carpeta '{CARPETA_ENTRADA}'. Copia ahí tus escaneados.")

for nombre in pdfs:
    ruta = os.path.join(CARPETA_ENTRADA, nombre)
    doc_id, nuevo = bd.registrar_documento(con, ruta, paginas=contar_paginas(ruta))
    print(f"{doc_id}  {'registrado' if nuevo else 'ya existía'}:  {nombre}")

print()
mostrar_documentos()

## Sección 2 – Censurar y pasar a la carpeta `censurados/`

Censura todos los documentos en estado `registrado`. Por seguridad, un documento queda en estado **`revisar`** (y NO se envía a Claude) si:
- no se encontró ninguno de los textos a censurar (puede que el OCR no lo haya leído), o
- al volver a leer el PDF censurado, alguno de los textos todavía aparece.

Abre esos PDF en `censurados/`, revísalos y, si están bien, apruébalos con la celda de abajo.

In [ ]:
# === Sección 2: censura ===
textos = [t for t in TEXTOS_A_CENSURAR if t.strip()]
pendientes = bd.documentos_en_estado(con, "registrado")
if not pendientes:
    print("No hay documentos pendientes de censurar.")

for doc in pendientes:
    salida = os.path.join(CARPETA_CENSURADOS, f"{doc['id']}.pdf")
    print(f"{doc['id']}: censurando...")
    try:
        conteo = censurar_pdf(doc["ruta_original"], textos, salida,
                              tesseract_cmd=TESSERACT_CMD, dpi=DPI_SALIDA, margen=MARGEN)
    except Exception as e:
        bd.marcar_revisar(con, doc["id"], f"error al censurar: {e}")
        print(f"  ERROR: {e}")
        continue

    zonas = sum(conteo.values())
    for t, n in conteo.items():
        print(f"  '{t}': {n} aparición(es)")

    if zonas == 0:
        bd.marcar_revisar(con, doc["id"], "no se encontró ningún texto a censurar", salida, 0)
        print("  -> REVISAR: no se encontró nada que censurar")
        continue

    if VERIFICAR_CENSURA:
        print("  verificando...")
        quedan = verificar_censura(salida, textos, TESSERACT_CMD)
        if quedan:
            bd.marcar_revisar(con, doc["id"], f"todavía se lee: {quedan}", salida, zonas)
            print(f"  -> REVISAR: todavía se lee {quedan}")
            continue

    bd.marcar_censurado(con, doc["id"], salida, zonas)
    print(f"  -> censurado: {salida}")

print()
mostrar_documentos()

In [ ]:
# === Aprobar a mano un documento en 'revisar' (después de abrir su PDF y comprobar que está bien) ===
# Escribe los IDs aprobados, por ejemplo: APROBAR = ["DOC-000003"]
APROBAR = []

# Para volver a censurar uno (por ejemplo, tras añadir textos a TEXTOS_A_CENSURAR):
# REPETIR = ["DOC-000003"]
REPETIR = []

for doc_id in APROBAR:
    bd.aprobar_censura(con, doc_id)
    print(f"{doc_id} aprobado: pasa a la sección 3")
for doc_id in REPETIR:
    bd.reiniciar(con, doc_id)
    print(f"{doc_id} vuelve a 'registrado': ejecuta otra vez la sección 2")
if APROBAR or REPETIR:
    print()
    mostrar_documentos()

## Sección 3 – Extraer los datos con Claude

Envía a Claude cada documento en estado `censurado` (solo el PDF censurado) y guarda los campos de `CAMPOS` en la base de datos.

**API key (solo la primera vez):** créala en https://console.anthropic.com → *API Keys*. Luego, en PowerShell:
```
setx ANTHROPIC_API_KEY "tu-api-key"
```
Cierra y vuelve a abrir VS Code para que la reconozca. **No la pegues en el notebook** ni la subas a GitHub.

Cada documento consume tokens de tu cuenta de Anthropic (se cobra por uso).

In [ ]:
# === Sección 3: extracción con Claude ===
import anthropic
from ekatena.extraccion_claude import extraer_datos

pendientes = bd.documentos_en_estado(con, "censurado")
if not pendientes:
    print("No hay documentos censurados pendientes de extraer.")
else:
    client = anthropic.Anthropic()

for doc in pendientes:
    print(f"{doc['id']}: enviando a Claude...")
    try:
        datos = extraer_datos(doc["ruta_censurado"], CAMPOS, client=client)
    except anthropic.AuthenticationError:
        print("  ERROR: API key inválida o no configurada (ANTHROPIC_API_KEY). Se detiene la sección 3.")
        break
    except anthropic.RateLimitError:
        print("  Límite de uso alcanzado. Espera un minuto y vuelve a ejecutar la celda.")
        break
    except anthropic.APIConnectionError:
        print("  ERROR: sin conexión con la API. Revisa tu internet y vuelve a ejecutar la celda.")
        break
    except (anthropic.APIStatusError, RuntimeError, ValueError) as e:
        bd.marcar_revisar(con, doc["id"], f"error en extracción: {e}")
        print(f"  ERROR: {e}")
        continue

    bd.guardar_extraccion(con, doc["id"], datos)
    for campo, valor in datos.items():
        print(f"  {campo}: {valor}")

print()
mostrar_documentos()

## Resultados

Tabla con los datos extraídos de todos los documentos (una fila por documento).

In [ ]:
# === Ver los datos extraídos ===
import json

filas = con.execute(
    "SELECT id, nombre_archivo, datos_json FROM documentos WHERE estado = 'extraido' ORDER BY num"
).fetchall()
if not filas:
    print("Todavía no hay datos extraídos.")
for f in filas:
    print(f"{f['id']}  ({f['nombre_archivo']})")
    for campo, valor in json.loads(f["datos_json"]).items():
        print(f"    {campo:<28} {valor}")
    print()

# Para exportar a Excel/CSV:
# import pandas as pd
# pd.read_sql("SELECT * FROM datos_extraidos", con).to_csv("datos_extraidos.csv", index=False)